# Backbone comparison: does the mediator beat the population model?

One question, asked of every backbone: **at n=100 ratings per user, does the
7-dimensional emotion mediator (Hybrid) beat the population GIAA model?**

Two anchoring schemes are reported side by side, because they do not answer
the same question:

| | how the personal head is anchored | anchored to |
|---|---|---|
| **B** | shrink the weights toward `w_pop` | the population fit **in that mediator's own space** |
| **C** | fit on the residual `y - y_pop` | the **same true GIAA model** for every mediator |

Under B each mediator is anchored to a different baseline, so Direct (anchored
to the full 512-d GIAA) and Hybrid (anchored to a weaker 7-d one) do not start
from the same place. C is the fair comparison. The tables below make the
consequence visible.

Everything is recomputed from `output/raw_all.csv`; no number is typed in.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.utils.metrics import tost_equivalence

LABEL = {'clip': 'CLIP frozen', 'clip_ft': 'CLIP-ft (score)',
         'clip_ft_emo': 'CLIP-ft (emotion)', 'qwen4b': 'Qwen3-VL 4B',
         'qwen8b': 'Qwen3-VL 8B'}
ORDER = list(LABEL)
UNIT = ['fold', 'domain', 'user_id']

ALL = pd.read_csv('../output/raw_all.csv', low_memory=False)
ALL = ALL[ALL['head'] == 'ridge']
raw = ALL[ALL['n_train'] == 100]
print(f'{len(raw):,} rows at n=100 | backbones: {sorted(raw.backbone.unique())}')


40,603 rows at n=100 | backbones: ['clip', 'clip_ft', 'clip_ft_emo', 'qwen4b', 'qwen8b']


## Building one row per backbone

A *unit* is one user in one domain (387 of them). Scores are averaged over the
3 seeds first, so each unit contributes once; the Wilcoxon test is then paired
across units, which makes 'Hybrid beats population' a statement about users
rather than about the mean.


In [2]:
def per_unit(frame, backbone, variant, n=None):
    g = frame[(frame.backbone == backbone) & (frame.variant == variant)]
    if n is not None:
        g = g[g.n_train == n]
    if g.empty:
        return None
    g = g.groupby(UNIT + ['mediator'], as_index=False)['srocc'].mean()
    p = g.pivot_table(index=UNIT, columns='mediator', values='srocc')
    need = {'emotion', 'identity', 'population'}
    if not need <= set(p.columns):
        return None
    return p.dropna(subset=list(need))


def table(variant):
    rows = []
    for bb in ORDER:
        p = per_unit(raw, bb, variant)
        if p is None:
            continue
        H, D, P = p['emotion'], p['identity'], p['population']
        rows.append({'backbone': LABEL[bb], 'units': len(p),
                     'Population': P.mean(),
                     'Hybrid (7-d)': H.mean(),
                     'Direct (512-d)': D.mean(),
                     'Hybrid - Pop': (H - P).mean(),
                     'p (Wilcoxon)': wilcoxon(H, P)[1],
                     'Hybrid wins': (H > P).mean(),
                     'Direct - Hybrid gap': (D - H).mean()})
    return pd.DataFrame(rows).set_index('backbone')


FMT = {'Population': '{:.4f}', 'Hybrid (7-d)': '{:.4f}',
       'Direct (512-d)': '{:.4f}', 'Hybrid - Pop': '{:+.4f}',
       'p (Wilcoxon)': '{:.4f}', 'Hybrid wins': '{:.1%}',
       'Direct - Hybrid gap': '{:+.4f}'}


def show(df, caption):
    return (df.style.format(FMT)
            .background_gradient(subset=['Hybrid - Pop'], cmap='RdYlGn',
                                 vmin=-0.02, vmax=0.02)
            .apply(lambda s: ['font-weight:bold' if v < .05 else 'color:#999'
                              for v in s], subset=['p (Wilcoxon)'])
            .set_caption(caption))


## Table 1 - Anchor B

Each mediator is anchored to the population model fitted *in its own space*.


In [3]:
tB = table('B')
show(tB, 'Anchor B, n=100, 3 seeds, ridge head. Bold p < .05.')


,units,Population,Hybrid (7-d),Direct (512-d),Hybrid - Pop,p (Wilcoxon),Hybrid wins,Direct - Hybrid gap
backbone,,,,,,,,
CLIP frozen,386,0.4174,0.4233,0.4576,+0.0059,0.2941,50.3%,+0.0343
CLIP-ft (score),386,0.4140,0.4265,0.4506,+0.0125,0.0120,55.7%,+0.0241
CLIP-ft (emotion),387,0.4074,0.4160,0.4489,+0.0086,0.0919,49.9%,+0.0329
Qwen3-VL 4B,386,0.4359,0.4384,0.4695,+0.0024,0.8128,47.4%,+0.0312
Qwen3-VL 8B,386,0.4395,0.4490,0.4776,+0.0095,0.1805,52.8%,+0.0285


In [4]:
print(f'Hybrid significantly beats population on '
      f'{(tB["p (Wilcoxon)"] < .05).sum()} of {len(tB)} backbones')


Hybrid significantly beats population on 1 of 5 backbones


## Table 2 - Anchor C

Every mediator is anchored to the **same** true GIAA model.


In [5]:
tC = table('C')
show(tC, 'Anchor C, n=100, 3 seeds, ridge head. Bold p < .05.')


,units,Population,Hybrid (7-d),Direct (512-d),Hybrid - Pop,p (Wilcoxon),Hybrid wins,Direct - Hybrid gap
backbone,,,,,,,,
CLIP frozen,386,0.4174,0.4363,0.4576,+0.0189,0.0000,59.1%,+0.0213
CLIP-ft (score),386,0.4140,0.4269,0.4506,+0.0129,0.0011,55.7%,+0.0237
CLIP-ft (emotion),387,0.4074,0.4247,0.4489,+0.0172,0.0000,58.9%,+0.0243
Qwen3-VL 4B,386,0.4359,0.4486,0.4695,+0.0127,0.0024,56.7%,+0.0209
Qwen3-VL 8B,386,0.4395,0.4603,0.4776,+0.0208,0.0000,60.6%,+0.0172


In [6]:
print(f'Hybrid significantly beats population on '
      f'{(tC["p (Wilcoxon)"] < .05).sum()} of {len(tC)} backbones')


Hybrid significantly beats population on 5 of 5 backbones


## Why C rather than B

Four checks. The first and the last are the strong ones; the middle two are
supporting evidence, and check 2 in particular is a difference of degree.


In [7]:
# 1. Direct is mathematically the same model under B and C. For the identity
#    mediator the transform is the identity, so w_pop IS the true GIAA and the
#    two residuals coincide. Were the implementation wrong, these would differ.
chk = pd.DataFrame({'Direct under B': tB['Direct (512-d)'],
                    'Direct under C': tC['Direct (512-d)'],
                    'Hybrid under B': tB['Hybrid (7-d)'],
                    'Hybrid under C': tC['Hybrid (7-d)']})
chk['Direct identical'] = np.isclose(chk['Direct under B'],
                                     chk['Direct under C'], atol=1e-12)
chk['Hybrid identical'] = np.isclose(chk['Hybrid under B'],
                                     chk['Hybrid under C'], atol=1e-12)
chk.round(4)


,Direct under B,Direct under C,Hybrid under B,Hybrid under C,Direct identical,Hybrid identical
backbone,,,,,,
CLIP frozen,0.4576,0.4576,0.4233,0.4363,True,False
CLIP-ft (score),0.4506,0.4506,0.4265,0.4269,False,False
CLIP-ft (emotion),0.4489,0.4489,0.4160,0.4247,True,False
Qwen3-VL 4B,0.4695,0.4695,0.4384,0.4486,True,False
Qwen3-VL 8B,0.4776,0.4776,0.4490,0.4603,True,False


In [8]:
# 2. How often does Hybrid sit BELOW the population row it is anchored on?
#    Under B this is a symptom of the anchor being the weaker 7-d fit. Under C
#    it can still happen at small n, because a correction fitted on 10 ratings
#    can hurt even when the anchor is right -- so this is a difference of
#    degree, not the clean impossibility argument it first looks like.
for variant in ('B', 'C'):
    bad = []
    for bb in ORDER:
        for n in (10, 25, 50, 100):
            p = per_unit(ALL, bb, variant, n=n)
            if p is None:
                continue
            if p['emotion'].mean() < p['population'].mean():
                bad.append(f'{LABEL[bb]} n={n}')
    print(f'anchor {variant}: Hybrid below population in {len(bad)} cells'
          + (': ' + ', '.join(bad) if bad else ''))


anchor B: Hybrid below population in 6 cells: CLIP frozen n=10, CLIP frozen n=25, CLIP frozen n=50, CLIP-ft (emotion) n=10, CLIP-ft (emotion) n=25, CLIP-ft (emotion) n=50


anchor C: Hybrid below population in 4 cells: CLIP frozen n=10, CLIP frozen n=25, CLIP-ft (emotion) n=10, Qwen3-VL 8B n=10


In [9]:
# 3. The Direct-Hybrid gap shrinks once both are anchored to the same model.
gap = pd.DataFrame({'gap under B': tB['Direct - Hybrid gap'],
                    'gap under C': tC['Direct - Hybrid gap']})
gap['shrinkage'] = 1 - gap['gap under C'] / gap['gap under B']
gap.style.format({'gap under B': '{:+.4f}', 'gap under C': '{:+.4f}',
                  'shrinkage': '{:.0%}'})


,gap under B,gap under C,shrinkage
backbone,,,
CLIP frozen,+0.0343,+0.0213,38%
CLIP-ft (score),+0.0241,+0.0237,2%
CLIP-ft (emotion),+0.0329,+0.0243,26%
Qwen3-VL 4B,+0.0312,+0.0209,33%
Qwen3-VL 8B,+0.0285,+0.0172,40%


In [10]:
# 4. B has no defined form for an MLP head: it shrinks the weight vector
#    toward w_pop, and an MLP has no weight vector to shrink. C is a residual
#    fit and applies to any head, so MLP rows can only exist under C.
print('B: linear heads only (ridge / lasso / elastic)')
print('C: any head, including MLP')


B: linear heads only (ridge / lasso / elastic)
C: any head, including MLP


## What the Direct-Hybrid gap is, and is not

Under anchor C both rows start from the same population prediction and learn a
personal correction on top. Direct's correction has 512 free parameters per
user; Hybrid's has 7. The gap is a capacity difference by design, not evidence
that the mediator discards useful signal.

The claim worth testing is the bounded one: *is the 7-parameter head close
enough to the 512-parameter one to be worth its interpretability?* That is an
equivalence question. A non-significant difference test cannot answer it --
failing to detect a difference is also what an underpowered study looks like.
TOST (two one-sided tests) answers it directly: it rejects 'the difference is
at least delta' from both sides, so passing means the true difference lies
inside (-delta, +delta).

**delta must be fixed on substantive grounds before looking at the output.**
Several values are shown to expose the sensitivity, not to pick one that passes.


In [11]:
eq = []
for bb in ORDER:
    for n in (10, 25, 50, 100):
        p = per_unit(ALL, bb, 'C', n=n)
        if p is None:
            continue
        row = {'backbone': LABEL[bb], 'n': n,
               'Hybrid - Direct': (p['emotion'] - p['identity']).mean()}
        for d in (0.01, 0.02, 0.03):
            t = tost_equivalence(p['emotion'], p['identity'], delta=d)
            row[f'equivalent (d={d})'] = 'yes' if t['equivalent'] else 'no'
        eq.append(row)

eq = pd.DataFrame(eq).set_index(['backbone', 'n'])
eq.style.format({'Hybrid - Direct': '{:+.4f}'})


## Caveat that has to travel with the CLIP-ft (emotion) row

That backbone was fine-tuned to predict the seven emotions directly, so its
512-d features are already an emotion representation. 'Direct' on it is not a
mediator-free control -- it is a 512-d emotion representation competing with a
7-d one, and the wider gap follows from that rather than from anything about
the mediator. Its gap should not be read against the other backbones'.


---

# One backbone across every support size and anchor

The n=100 tables above are one column of a curve. This section fixes the
backbone and varies both the support size and the anchor -- all four, including
`A`, which the n=100 tables omit.

`gap` is Direct - Hybrid; `H - Pop` is Hybrid - population; `p` is the paired
Wilcoxon of Hybrid against population across the 387 user-domain units.

An anchor that has not been run for a backbone is reported as missing rather
than left blank, so an incomplete grid cannot be mistaken for a finished one.


In [12]:
ANCHORS = ['plain', 'A', 'B', 'C']
SIZES = [10, 25, 50, 100]


def by_support(backbone, variant):
    rows = []
    for n in SIZES:
        p = per_unit(ALL, backbone, variant, n=n)
        if p is None:
            continue
        H, D, P = p['emotion'], p['identity'], p['population']
        rows.append({'n_train': n,
                     'Hybrid': H.mean(), 'Direct': D.mean(),
                     'population': P.mean(),
                     'gap': (D - H).mean(),
                     'H - Pop': (H - P).mean(),
                     'p': wilcoxon(H, P)[1],
                     'Hybrid wins': (H > P).mean()})
    return pd.DataFrame(rows).set_index('n_train') if rows else None


SFMT = {'Hybrid': '{:.4f}', 'Direct': '{:.4f}', 'population': '{:.4f}',
        'gap': '{:+.4f}', 'H - Pop': '{:+.4f}', 'p': '{:.4f}',
        'Hybrid wins': '{:.1%}'}


def anchor_panels(backbone):
    """One styled table per anchor, or a note saying what is missing."""
    from IPython.display import display, Markdown
    display(Markdown(f'### {LABEL[backbone]}'))
    for v in ANCHORS:
        t = by_support(backbone, v)
        if t is None:
            display(Markdown(f'**anchor {v}** - not run yet'))
            continue
        missing = sorted(set(SIZES) - set(t.index))
        note = f' (missing n={missing})' if missing else ''
        display(t.style.format(SFMT)
                .background_gradient(subset=['H - Pop'], cmap='RdYlGn',
                                     vmin=-0.05, vmax=0.05)
                .apply(lambda s: ['font-weight:bold' if v_ < .05 else 'color:#999'
                                  for v_ in s], subset=['p'])
                .set_caption(f'{LABEL[backbone]} x ridge - anchor {v}{note}'))


## CLIP-ft (emotion)


In [13]:
anchor_panels('clip_ft_emo')


### CLIP-ft (emotion)

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3214,0.3101,0.4074,-0.0113,-0.0860,0.0000,33.1%
25,0.3705,0.3810,0.4074,+0.0105,-0.0370,0.0000,37.0%
50,0.3932,0.4154,0.4074,+0.0222,-0.0143,0.0226,44.4%
100,0.4151,0.4396,0.4074,+0.0245,+0.0077,0.1624,50.6%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3306,0.3121,0.4074,-0.0185,-0.0768,0.0000,36.7%
25,0.3772,0.3817,0.4074,+0.0045,-0.0303,0.0000,41.1%
50,0.4026,0.4157,0.4074,+0.0132,-0.0049,0.9692,50.9%
100,0.4252,0.4401,0.4074,+0.0149,+0.0178,0.0002,55.6%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3935,0.4110,0.4074,+0.0175,-0.0139,0.0000,40.6%
25,0.3964,0.4226,0.4074,+0.0262,-0.0110,0.0010,42.9%
50,0.3993,0.4328,0.4074,+0.0336,-0.0082,0.1474,46.5%
100,0.4160,0.4489,0.4074,+0.0329,+0.0086,0.0919,49.9%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.4063,0.4110,0.4074,+0.0047,-0.0011,0.9468,42.4%
25,0.4078,0.4226,0.4074,+0.0148,+0.0004,0.2380,51.7%
50,0.4137,0.4328,0.4074,+0.0192,+0.0062,0.0008,53.5%
100,0.4247,0.4489,0.4074,+0.0243,+0.0172,0.0000,58.9%


## CLIP-ft (score)

Cells that have not been run yet are reported as missing rather than left
blank, so an incomplete row cannot be mistaken for a finished one.


In [14]:
anchor_panels('clip_ft')


### CLIP-ft (score)

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3294,0.3079,0.4138,-0.0216,-0.0844,0.0000,34.1%
25,0.3922,0.3874,0.4138,-0.0048,-0.0216,0.0010,44.2%
50,0.4093,0.4257,0.4138,+0.0163,-0.0045,0.1821,48.3%
100,0.4254,0.4468,0.4140,+0.0214,+0.0115,0.0881,52.6%


**anchor A** - not run yet

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
100,0.4265,0.4506,0.4140,+0.0241,+0.0125,0.0120,55.7%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
100,0.4269,0.4506,0.4140,+0.0237,+0.0129,0.0011,55.7%


## The other backbones, for reference


In [15]:
for bb in ('clip', 'qwen4b', 'qwen8b'):
    anchor_panels(bb)


### CLIP frozen

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3255,0.2231,0.4159,-0.1024,-0.0904,0.0000,29.5%
25,0.3795,0.2929,0.4159,-0.0866,-0.0364,0.0000,37.5%
50,0.4038,0.3473,0.4159,-0.0565,-0.0121,0.0201,45.2%
100,0.4187,0.3998,0.4159,-0.0189,+0.0028,0.8416,49.1%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3391,0.2291,0.4159,-0.1100,-0.0768,0.0000,33.1%
25,0.3900,0.2989,0.4159,-0.0911,-0.0259,0.0004,42.6%
50,0.4151,0.3543,0.4159,-0.0608,-0.0008,0.8402,48.1%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.4041,0.4196,0.4159,+0.0156,-0.0118,0.0000,38.8%
25,0.4047,0.4312,0.4159,+0.0265,-0.0112,0.0026,44.7%
50,0.4138,0.4404,0.4159,+0.0266,-0.0021,0.1301,46.0%
100,0.4233,0.4576,0.4174,+0.0343,+0.0059,0.2941,50.3%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.4138,0.4196,0.4159,+0.0059,-0.0021,0.5660,44.7%
25,0.4152,0.4312,0.4159,+0.0161,-0.0008,0.4594,47.3%
50,0.4232,0.4404,0.4159,+0.0172,+0.0073,0.1225,48.8%
100,0.4363,0.4576,0.4174,+0.0213,+0.0189,0.0000,59.1%


...

### Qwen3-VL 8B

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.3550,0.2460,0.4392,-0.1090,-0.0843,0.0000,28.9%
25,0.4033,0.3148,0.4392,-0.0885,-0.0359,0.0000,36.7%
50,0.4283,0.3806,0.4392,-0.0478,-0.0109,0.0094,43.4%
100,0.4461,0.4262,0.4394,-0.0198,+0.0067,0.6097,51.7%


**anchor A** - not run yet

,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
100,0.4490,0.4776,0.4395,+0.0285,+0.0095,0.1805,52.8%


,Hybrid,Direct,population,gap,H - Pop,p,Hybrid wins
n_train,,,,,,,
10,0.4333,0.4374,0.4392,+0.0041,-0.0060,0.9674,36.2%
100,0.4603,0.4776,0.4395,+0.0172,+0.0208,0.0000,60.6%


Reading it: under **plain**, the 7-d mediator is ahead at every small support
size on the frozen backbones, and the advantage decays as n grows -- exactly the
shape the low-data argument predicts. Under **C**, the population anchor already
supplies what the mediator was supplying, so the remaining difference is the
raw capacity of the correction and Direct leads throughout.

The two fine-tuned backbones behave differently under plain, because their
features are already task-aligned -- see the caveat at the end of this notebook.
